# PIP-FUCCI cell-cycle phase analysis

Detects G1/S and S/G2 transitions from single-cell PIP-FUCCI traces (CDT1 = green, Geminin = red), applies quality control, manual curation and IQR outlier removal, and reports G1, S and G2/M durations.

**Input:** one combined, normalized CSV per genotype (columns: `Cell_ID`, `Time` in hours, `CDT1_Signal`, `Geminin_Signal`, `Green_Raw`, `Red_Raw`).

**How to use:**
1. The input CSVs are included in `data/` (file names below).
2. Choose a genotype in the settings cell.
3. Run all cells. Outputs are written to `results/<genotype>/`.

By default the manual exclusions used in the paper (`exclusions/<genotype>.json`) are applied, so the published results are reproduced without repeating the visual review. Set `REDO_MANUAL_REVIEW = True` to review the cells yourself instead.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from pip_fucci_analyzer import ImprovedPIPFUCCIAnalyzer
from improved_workflow import (
    step1_initial_analysis,
    step2_manual_review_with_save,
    extract_plot_data,
    extract_population_summary_data,
    save_exclusions,
)

## Settings

These are the settings used for all four genotypes in the paper.

Fixed inside `improved_workflow.step1_initial_analysis` (not changed here):
- Raw-signal QC on; thresholds = 10th percentile of per-cell maximum raw intensity, determined per dataset
- Sustained-drop / sustained-rise windows for G1/S and S/G2 calls: 0.5 h
- Outlier removal: 1.5 x IQR on S and G2/M durations (applied after manual review)

In [ ]:
GENOTYPE = 'WT'   # one of: 'WT', 'p53KO', 'Per2KO', 'dKO'

REDO_MANUAL_REVIEW = False   # False = apply the published exclusions; True = interactive review

INPUT_FILES = {
    'WT':     'data/combined_all_conditions_WT.csv',
    'p53KO':  'data/combined_all_conditions_p53KO.csv',
    'Per2KO': 'data/combined_all_conditions_Per2KO.csv',
    'dKO':    'data/combined_all_conditions_dKO.csv',
}

DATASET_CONFIG = {
    'name': GENOTYPE,
    'csv_path': INPUT_FILES[GENOTYPE],
    'output_dir': f'results/{GENOTYPE}',
    'min_g1_duration': 2.0,
    'sg2_sensitivity': 'medium',
    'plot_size': (12, 6),   # figure size for the interactive review
}

EXCLUSION_FILE = Path('exclusions') / f'{GENOTYPE}.json'

## 1. Transition detection, QC, manual curation, outliers, figures

In [ ]:
# Step 1: detect transitions and run automatic QC on every cell
analyzer = step1_initial_analysis(DATASET_CONFIG, load_saved_exclusions=False)

# Step 2: manual curation
if REDO_MANUAL_REVIEW:
    exclusions, kept = step2_manual_review_with_save(analyzer, DATASET_CONFIG)
else:
    with open(EXCLUSION_FILE) as f:
        exclusions = json.load(f)['exclusions']
    print(f'Applying {len(exclusions)} published manual exclusions from {EXCLUSION_FILE}')
    analyzer.exclude_cells(exclusions, reason='MANUAL_VISUAL_REVIEW')
    save_exclusions(analyzer, exclusions, reason='MANUAL_VISUAL_REVIEW')

# Step 3: outlier detection (writes results_no_outliers.csv = final table)
analyzer.detect_phase_outliers(iqr_multiplier=1.5)

# Step 4: figures
analyzer.plot_phase_durations_violin(qc_passed_only=True)
analyzer.plot_population_summary(qc_passed_only=True)

# Step 5: export plot-ready tables
plot_data = extract_plot_data(analyzer)
population_data = extract_population_summary_data(analyzer)

## 2. Export phase durations (e.g. for GraphPad Prism)

In [ ]:
prism_data = plot_data['cells'][['Cell_ID', 'G1_duration', 'S_phase_duration', 'G2M_duration']]
prism_data.to_csv(analyzer.output_dir / 'for_prism.csv', index=False)
print(plot_data['statistics'])

## 3. Phase-duration correlations

In [ ]:
output_dir = Path(DATASET_CONFIG['output_dir'])
results = pd.read_csv(output_dir / 'results_no_outliers.csv')
complete = results[results['Has_complete_S_phase'] == True].copy()

# G2/M duration = end of track - S/G2 transition
complete['G2M_duration'] = complete['End_time'] - complete['SG2_time']
complete = complete[complete['G2M_duration'] > 0]

print(f'Cells with complete S phase and valid G2/M: {len(complete)}')


def scatter_with_regression(ax, x, y, color, xlabel, ylabel, title):
    valid = pd.DataFrame({'x': x, 'y': y}).dropna()
    ax.scatter(valid['x'], valid['y'],
               alpha=0.5, s=50, color=color,
               edgecolors='white', linewidth=0.5)
    if len(valid) >= 3:
        slope, intercept, r, p, _ = stats.linregress(valid['x'], valid['y'])
        x_line = np.linspace(valid['x'].min(), valid['x'].max(), 100)
        ax.plot(x_line, slope * x_line + intercept,
                color='#e74c3c', linewidth=1.5, linestyle='--', alpha=0.8)
        p_str = f'{p:.3f}' if p >= 0.001 else '<0.001'
        ax.text(0.05, 0.95,
                f'R² = {r**2:.3f}\np = {p_str}',
                transform=ax.transAxes, fontsize=9,
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.2)


fig, axes = plt.subplots(1, 3, figsize=(15, 4))

scatter_with_regression(
    axes[0],
    complete['G1_duration'], complete['G2M_duration'],
    color='#2ecc71',
    xlabel='G1 Duration (h)', ylabel='G2/M Duration (h)',
    title='G1 vs G2/M Duration')

scatter_with_regression(
    axes[1],
    complete['G1_duration'], complete['S_phase_duration'],
    color='#f39c12',
    xlabel='G1 Duration (h)', ylabel='S Duration (h)',
    title='G1 vs S Duration')

scatter_with_regression(
    axes[2],
    complete['S_phase_duration'], complete['G2M_duration'],
    color='#3498db',
    xlabel='S Duration (h)', ylabel='G2/M Duration (h)',
    title='S vs G2/M Duration')

plt.suptitle(f'Phase Duration Correlations (n={len(complete)} cells)',
             fontsize=13, fontweight='bold')
plt.tight_layout()

save_path = output_dir / 'phase_correlations.svg'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f'Saved: {save_path}')
plt.show()